# 4. Simulación de un robot aéreo en 3D

Se trabajará con el modelo no lineal de un robot aéreo (quadcópter) en 3 dimensiones. Se controlará de manera directa el empuje ("thrust") solo a modo de visualización de la simulación.

El estado de este robot se puede describir mediante el siguiente vector:

$$X = [x, \,y, \,z, \,\phi, \,\theta, \,\psi, \,\dot{x}, \,\dot{y}, \,\dot{z}, \,p, \,q, \,r]$$

donde los tres primeros términos son la posición, los siguientes tres términos representan la orientación a través de ángulos de roll, pitch, yaw, los siguientes tres términos son la velocidad lineal en el sistema de referencia inercial, y los tres últimos términos representan la velocidad angular en el sistema de referencia del cuerpo (local).

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

from Robot3D import (RobotAereo3D, Rrpy, convert_control_to_motors,
                     AnimacionRobot3D, plot_trajectory)

### Simulación 

Se utiliza un controlador en lazo abierto que asigna por defecto cierta velocidad angular $\omega_i$ a cada rotor, generando una fuerza de empuje que brinda un movimiento arbitrario.

In [ ]:
# Tiempo de simulación
tiempo_total = 20.0
dt = 0.01
t = np.linspace(0.0, tiempo_total, int(tiempo_total/dt))

# Robot simulado
robot = RobotAereo3D()

# Parámetros "estimados" del robot
m = 0.5; g = 9.81; kf = 1.0; km = 1.0;

# Estado inicial del robot con altura 0 y velocidad 0
X0 = np.array([0.0, 0.0, 0.0, 0.1, 0.0, 0.0,
              0.0, 0.0, 0.0, 0.0, 0.0, 0.0])
robot.set_state(X0)

# Historial
Xt = np.zeros((t.size, 12))
Xt[0,:] = robot.X

# Ejecución del vuelo
for i in range(0, t.size-1):

    # Controlador
    w1 = np.sqrt((1/4*m*g)/kf) + 0.1
    w2 = np.sqrt((1/4*m*g)/kf) + 0.1
    w3 = np.sqrt((1/4*m*g)/kf) + 0.1
    w4 = np.sqrt((1/4*m*g)/kf) + 0.1

    # Simulación
    robot.set_control(w1, w2, w3, w4)
    robot_state = robot.simulate(dt)

    # Generación del historial de las posiciones verticales
    Xt[i+1,:] = robot_state

# Recuperación del estado
traj_x = Xt[:,0]; traj_y = Xt[:,1]; traj_z = Xt[:,2]
traj_phi = Xt[:,3]; traj_theta = Xt[:,4]; traj_psi = Xt[:,5]
traj_dx = Xt[:,6]; traj_dy = Xt[:,7]; traj_dz = Xt[:,8]

### Trayectoria

Se grafica la trayectoria desde el punto inicial (en verde) hasta el punto final (en rojo)

In [ ]:
fig = plot_trajectory(traj_x, traj_y, traj_z)
fig.show()

Se mostrará una animación del movimiento del robot. Se puede utilizar los botones de la parte inferior para reproducir el movimiento.

In [ ]:
# Mostrar una animación del vuelo del robot
ts = 10              # Factor de muestreo
porcentaje = 0.2     # Porcentaje (0 a 1) del tiempo total
tf = int(np.floor(porcentaje*(tiempo_total/dt)))

traj = {'time': t[0:tf:ts],
        'position': np.array([traj_x[0:tf:ts], traj_y[0:tf:ts], traj_z[0:tf:ts]]).T,
        'orientation': np.array([traj_phi[0:tf:ts], traj_theta[0:tf:ts], traj_psi[0:tf:ts]]).T}

sim = AnimacionRobot3D()
sim.run_simulation(trajectory_data = traj)

Gráfico 2D de la trayectoria de vuelo

In [ ]:
plt.figure(figsize=(6,4))

plt.plot(traj_y, traj_z, linestyle='-', color='blue')
plt.title('Trayectoria de vuelo').set_fontsize(10)
plt.xlabel('$x$ [$m$]').set_fontsize(10); plt.ylabel('$z$ [$m$]').set_fontsize(10)
plt.xticks(fontsize=10); plt.yticks(fontsize=10)
plt.gca().invert_yaxis()
plt.axis('equal'); plt.grid()
plt.show()